In [11]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 13, Finished, Available, Finished, False)


# Telemetry DQ validation



In [12]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.dq_telemetry AS

        WITH duplicate_check AS
        (
            SELECT
                *,
                COUNT(*) OVER
                (
                    PARTITION BY
                        raw_timestamp,
                        timestamp,
                        site_id,
                        building_id,
                        asset_id,
                        sensor_id,
                        temperature,
                        humidity,
                        pressure,
                        vibration,
                        power_consumption,
                        operating_mode
                ) AS duplicate_count
            FROM dbo.silver_telemetry
        ),

        asset_check AS
        (
            SELECT DISTINCT
                TRIM(asset_id) AS asset_id
            FROM dbo.silver_asset_metadata
            WHERE asset_id IS NOT NULL
              AND TRIM(asset_id) <> ''
        )

        SELECT
            t.*,

            CASE

                WHEN t.raw_timestamp IS NULL
                    THEN 'MISSING_TIMESTAMP'

                WHEN t.timestamp IS NULL
                    THEN 'INVALID_TIMESTAMP'

                WHEN t.site_id IS NULL
                     OR TRIM(t.site_id) = ''
                    THEN 'MISSING_SITE_ID'

                WHEN t.building_id IS NULL
                     OR TRIM(t.building_id) = ''
                    THEN 'MISSING_BUILDING_ID'

                WHEN t.asset_id IS NULL
                     OR TRIM(t.asset_id) = ''
                    THEN 'MISSING_ASSET_ID'

                WHEN t.sensor_id IS NULL
                     OR TRIM(t.sensor_id) = ''
                    THEN 'MISSING_SENSOR_ID'

                WHEN t.temperature IS NULL
                    THEN 'MISSING_TEMPERATURE'

                WHEN t.humidity IS NULL
                    THEN 'MISSING_HUMIDITY'

                WHEN t.pressure IS NULL
                    THEN 'MISSING_PRESSURE'

                WHEN t.vibration IS NULL
                    THEN 'MISSING_VIBRATION'

                WHEN t.power_consumption IS NULL
                    THEN 'MISSING_POWER_CONSUMPTION'

                WHEN t.operating_mode IS NULL
                    THEN 'MISSING_OPERATING_MODE'

                WHEN t.temperature < -50
                     OR t.temperature > 100
                    THEN 'OUTLIER_TEMPERATURE'

                WHEN t.humidity < 0
                     OR t.humidity > 100
                    THEN 'OUTLIER_HUMIDITY'

                WHEN t.pressure <= 0
                    THEN 'OUTLIER_PRESSURE'

                WHEN t.vibration < 0
                    THEN 'OUTLIER_VIBRATION'

                WHEN t.power_consumption < 0
                    THEN 'OUTLIER_POWER_CONSUMPTION'

                WHEN t.duplicate_count > 1
                    THEN 'DUPLICATE_RECORD'

                WHEN a.asset_id IS NULL
                    THEN 'INVALID_ASSET_ID'

                ELSE 'VALID'

            END AS validation_status

        FROM duplicate_check t

        LEFT JOIN asset_check a
            ON TRIM(t.asset_id) = a.asset_id
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5e6decb9-381c-46f3-8029-957bc6f14a22)

In [13]:
display(
    spark.sql("""
        SELECT
            validation_status,
            COUNT(*) AS record_count
        FROM dbo.dq_telemetry
        GROUP BY validation_status
        ORDER BY record_count DESC
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0eecef56-2dc0-479c-ac11-51cbe3738ece)

In [14]:
display(
    spark.sql("""
        SELECT *
        FROM dbo.dq_telemetry
        WHERE validation_status <> 'VALID'
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1f189f51-1447-4583-9979-16263f45e4c4)

# events DQ Validation

In [15]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.dq_events AS

        WITH duplicate_check AS
        (
            SELECT
                *,
                COUNT(*) OVER
                (
                    PARTITION BY
                        event_id,
                        timestamp,
                        asset_id,
                        event_type,
                        severity,
                        message
                ) AS duplicate_count
            FROM dbo.silver_events
        ),

        asset_check AS
        (
            SELECT DISTINCT
                TRIM(asset_id) AS asset_id
            FROM dbo.silver_asset_metadata
            WHERE asset_id IS NOT NULL
              AND TRIM(asset_id) <> ''
        )

        SELECT
            e.*,

            CASE

                -- Null / missing values
                WHEN e.event_id IS NULL
                     OR TRIM(e.event_id) = ''
                    THEN 'MISSING_EVENT_ID'

                WHEN e.timestamp IS NULL
                    THEN 'INVALID_TIMESTAMP'

                WHEN e.asset_id IS NULL
                     OR TRIM(e.asset_id) = ''
                    THEN 'MISSING_ASSET_ID'

                WHEN e.event_type IS NULL
                     OR TRIM(e.event_type) = ''
                    THEN 'MISSING_EVENT_TYPE'

                WHEN e.severity IS NULL
                     OR TRIM(e.severity) = ''
                    THEN 'MISSING_SEVERITY'

                WHEN e.message IS NULL
                     OR TRIM(e.message) = ''
                    THEN 'MISSING_MESSAGE'

                -- Duplicate events
                WHEN e.duplicate_count > 1
                    THEN 'DUPLICATE_EVENT'

                ELSE 'VALID'

            END AS validation_status

        FROM duplicate_check e

        LEFT JOIN asset_check a
            ON TRIM(e.asset_id) = a.asset_id
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c2ef0e7-9907-43d8-b680-60f88e7128f5)

In [16]:
display(
    spark.sql("""
        SELECT
            validation_status,
            COUNT(*) AS record_count
        FROM dbo.dq_events
        GROUP BY validation_status
        ORDER BY record_count DESC
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 42f912e8-5dad-4c6e-ab73-a976a4f59b90)

In [17]:
display(
    spark.sql("""
        SELECT *
        FROM dbo.dq_events
        WHERE validation_status <> 'VALID'
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 52f04d8e-89ed-4567-8cb4-1f1949c932fb)

# assest_metadata DQ Validation

In [18]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.dq_asset_metadata AS

        WITH duplicate_check AS
        (
            SELECT
                *,
                COUNT(*) OVER
                (
                    PARTITION BY
                        asset_id
                ) AS duplicate_count
            FROM dbo.silver_asset_metadata
        )

        SELECT
            a.*,

            CASE

                WHEN a.asset_id IS NULL
                     OR TRIM(a.asset_id) = ''
                    THEN 'MISSING_ASSET_ID'

                WHEN a.asset_name IS NULL
                     OR TRIM(a.asset_name) = ''
                    THEN 'MISSING_ASSET_NAME'

                WHEN a.asset_type IS NULL
                     OR TRIM(a.asset_type) = ''
                    THEN 'MISSING_ASSET_TYPE'

                WHEN a.manufacturer IS NULL
                     OR TRIM(a.manufacturer) = ''
                    THEN 'MISSING_MANUFACTURER'

                WHEN a.installation_date IS NULL
                    THEN 'INVALID_INSTALLATION_DATE'

                WHEN a.site_id IS NULL
                     OR TRIM(a.site_id) = ''
                    THEN 'MISSING_SITE_ID'

                WHEN a.duplicate_count > 1
                    THEN 'DUPLICATE_ASSET'

                ELSE 'VALID'

            END AS validation_status

        FROM duplicate_check a
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 71ae0230-1332-41fd-b915-e2bbefd805b2)

In [19]:
display(
    spark.sql("""
        SELECT
            validation_status,
            COUNT(*) AS record_count
        FROM dbo.dq_asset_metadata
        GROUP BY validation_status
        ORDER BY record_count DESC
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a8dca872-fa20-4fca-ad28-c6dee6d62bf9)

In [20]:
display(
    spark.sql("""
        SELECT *
        FROM dbo.dq_asset_metadata
        WHERE validation_status <> 'VALID'
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8f57ee2b-6dcb-4870-846c-e5009df5b9c5)

# Telemetry quarantine

In [21]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.dq_quarantine_telemetry AS
        SELECT *
        FROM dbo.dq_telemetry
        WHERE validation_status <> 'VALID'
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 87ea7da0-66bb-44bc-af14-7955d6952878)

In [22]:
display(
    spark.sql("""
         
        SELECT *
        FROM dbo.dq_quarantine_telemetry 
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d04d3d83-dd46-4851-a92a-dcd605c68c3d)

# Events quarantine

In [23]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.dq_quarantine_events AS
        SELECT *
        FROM dbo.dq_events
        WHERE validation_status <> 'VALID'
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f0d55667-ef45-49ff-9bb9-343baba76dc3)

In [24]:
display(
    spark.sql("""
         
        SELECT *
        FROM dbo.dq_quarantine_events 
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7e272db7-fb3b-4c96-8144-bf944eaa4381)

# Asset metadata quarantine

In [25]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.dq_quarantine_asset_metadata AS
        SELECT *
        FROM dbo.dq_asset_metadata
        WHERE validation_status <> 'VALID'
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c4dadeb-f653-4877-b355-5e4975fbc88c)

In [26]:
display(
    spark.sql("""
         
        SELECT *
        FROM dbo.dq_quarantine_asset_metadata 
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 400d20c6-8b2a-432e-9468-2fdb850193a8)

# Overall  Missing record,Null,Invalid record is there

In [27]:
display(
    spark.sql("""
        SELECT 'telemetry' AS dataset,
               COUNT(*) AS quarantined_records
        FROM dbo.dq_quarantine_telemetry

        UNION ALL

        SELECT 'events',
               COUNT(*)
        FROM dbo.dq_quarantine_events

        UNION ALL

        SELECT 'assest_metadata',
               COUNT(*)
        FROM dbo.dq_quarantine_asset_metadata
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b3656a38-317e-43df-8697-9a29d78bbc9c)

 # Build Data Quality Framework 

 ## stores the QUALITY STATISTICS

#### This query creates a Data Quality audit/report table. It does not perform the validation itself. Instead,  DQ validation notebook calculates the results and stores those results in this table.

In [28]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dbo.dq_quality_report (
    run_id STRING,
    dataset_name STRING,
    check_name STRING,
    total_records BIGINT,
    failed_records BIGINT,
    passed_records BIGINT,
    failure_percentage DOUBLE,
    status STRING,
    validation_timestamp TIMESTAMP
)
""")

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 30, Finished, Available, Finished, False)

DataFrame[]

## **Generate automated quality reports.**

In [32]:
# =====================================================================
# DATA QUALITY FRAMEWORK - FINAL
# =====================================================================
# Purpose:
#   1. Validate Silver-layer data
#   2. Identify data quality issues
#   3. Calculate PASS / FAIL statistics
#   4. Store detailed results in dbo.dq_quality_report
#   5. Store run-level results in dbo.dq_quality_summary
#   6. Provide the final DQ status for pipeline control
#
# DQ REQUIREMENTS:
#   - Missing / incomplete records
#   - Duplicate events / records
#   - Schema / data-type violations
#   - Null values
#   - Outliers
#   - Late-arriving data
#
# DATASETS:
#   - telemetry
#   - events
#   - asset_metadata
#
# PIPELINE FLOW:
#
# Bronze
#    ↓
# Silver
#    ↓
# DQ Validation
#    ↓
# dq_quality_report
#    ↓
# dq_quality_summary
#    ↓
# PASS → Gold → NetworkX
# FAIL → DQ_Failed → Stop / Quarantine
# =====================================================================


# =====================================================================
# 1. IMPORTS AND UNIQUE RUN ID
# =====================================================================

import uuid

run_id = str(uuid.uuid4())


# =====================================================================
# 2. CREATE DETAILED DQ AUDIT TABLE
# =====================================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS dbo.dq_quality_report
(
    run_id STRING,
    dataset_name STRING,
    check_name STRING,

    total_records BIGINT,
    failed_records BIGINT,
    passed_records BIGINT,

    failure_percentage DOUBLE,
    status STRING,

    validation_timestamp TIMESTAMP
)
""")


# =====================================================================
# 3. REUSABLE FUNCTION TO STORE DQ RESULT
# =====================================================================

def write_dq_result(
    run_id,
    dataset_name,
    check_name,
    total_records,
    failed_records
):

    # Calculate passed records
    passed_records = max(total_records - failed_records, 0)

    # Calculate failure percentage
    failure_percentage = (
        (failed_records / total_records) * 100
        if total_records > 0
        else 0.0
    )

    # Check-level status
    status = (
        "PASS"
        if failed_records == 0
        else "FAIL"
    )

    spark.sql(f"""
        INSERT INTO dbo.dq_quality_report
        (
            run_id,
            dataset_name,
            check_name,
            total_records,
            failed_records,
            passed_records,
            failure_percentage,
            status,
            validation_timestamp
        )
        VALUES
        (
            '{run_id}',
            '{dataset_name}',
            '{check_name}',
            {total_records},
            {failed_records},
            {passed_records},
            {failure_percentage},
            '{status}',
            current_timestamp()
        )
    """)


# =====================================================================
# TELEMETRY VALIDATION
# =====================================================================

# ---------------------------------------------------------------------
# 4. TOTAL TELEMETRY RECORDS
# ---------------------------------------------------------------------

telemetry_total = spark.sql("""
    SELECT COUNT(*) AS total_records
    FROM dbo.silver_telemetry
""").collect()[0]["total_records"]


# ---------------------------------------------------------------------
# 5. INCOMPLETE TELEMETRY RECORD CHECK
# ---------------------------------------------------------------------
# A telemetry record is considered incomplete when mandatory
# business identifiers or timestamp are missing.
#
# NOTE:
# This detects incomplete records, not an unknown number of
# physically missing source rows.

telemetry_incomplete_failed = spark.sql("""
    SELECT COUNT(*) AS failed_records
    FROM dbo.silver_telemetry
    WHERE timestamp IS NULL
       OR site_id IS NULL
       OR building_id IS NULL
       OR asset_id IS NULL
       OR sensor_id IS NULL
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "telemetry",
    "INCOMPLETE_RECORD_CHECK",
    telemetry_total,
    telemetry_incomplete_failed
)


# ---------------------------------------------------------------------
# 6. TELEMETRY NULL VALUE CHECK
# ---------------------------------------------------------------------

telemetry_null_failed = spark.sql("""
    SELECT COUNT(*) AS failed_records
    FROM dbo.silver_telemetry
    WHERE timestamp IS NULL
       OR site_id IS NULL
       OR building_id IS NULL
       OR asset_id IS NULL
       OR sensor_id IS NULL
       OR temperature IS NULL
       OR humidity IS NULL
       OR pressure IS NULL
       OR vibration IS NULL
       OR power_consumption IS NULL
       OR operating_mode IS NULL
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "telemetry",
    "NULL_VALUE_CHECK",
    telemetry_total,
    telemetry_null_failed
)


# ---------------------------------------------------------------------
# 7. DUPLICATE TELEMETRY CHECK
# ---------------------------------------------------------------------
# Business key:
# timestamp + site + building + asset + sensor
#
# Only EXTRA duplicate records are counted as failures.
#
# Example:
# 3 identical records → 1 valid + 2 duplicate failures

telemetry_duplicate_failed = spark.sql("""
    SELECT
        COALESCE(SUM(duplicate_count - 1), 0) AS failed_records
    FROM
    (
        SELECT
            timestamp,
            site_id,
            building_id,
            asset_id,
            sensor_id,
            COUNT(*) AS duplicate_count
        FROM dbo.silver_telemetry
        GROUP BY
            timestamp,
            site_id,
            building_id,
            asset_id,
            sensor_id
        HAVING COUNT(*) > 1
    ) duplicates
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "telemetry",
    "DUPLICATE_CHECK",
    telemetry_total,
    telemetry_duplicate_failed
)





# ---------------------------------------------------------------------
# 9. TEMPERATURE OUTLIER CHECK
# ---------------------------------------------------------------------
# Business assumption:
# Valid temperature range = -20 to 100 degrees.

temperature_failed = spark.sql("""
    SELECT COUNT(*) AS failed_records
    FROM dbo.silver_telemetry
    WHERE temperature < -20
       OR temperature > 100
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "telemetry",
    "TEMPERATURE_OUTLIER_CHECK",
    telemetry_total,
    temperature_failed
)


# =====================================================================
# EVENTS VALIDATION
# =====================================================================

# ---------------------------------------------------------------------
# 10. TOTAL EVENT RECORDS
# ---------------------------------------------------------------------

events_total = spark.sql("""
    SELECT COUNT(*) AS total_records
    FROM dbo.silver_events
""").collect()[0]["total_records"]


# ---------------------------------------------------------------------
# 11. INCOMPLETE EVENT RECORD CHECK
# ---------------------------------------------------------------------

events_incomplete_failed = spark.sql("""
    SELECT COUNT(*) AS failed_records
    FROM dbo.silver_events
    WHERE event_id IS NULL
       OR timestamp IS NULL
       OR asset_id IS NULL
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "events",
    "INCOMPLETE_RECORD_CHECK",
    events_total,
    events_incomplete_failed
)


# ---------------------------------------------------------------------
# 12. DUPLICATE EVENT CHECK
# ---------------------------------------------------------------------
# event_id is the business key.

events_duplicate_failed = spark.sql("""
    SELECT
        COALESCE(SUM(duplicate_count - 1), 0) AS failed_records
    FROM
    (
        SELECT
            event_id,
            COUNT(*) AS duplicate_count
        FROM dbo.silver_events
        WHERE event_id IS NOT NULL
        GROUP BY event_id
        HAVING COUNT(*) > 1
    ) duplicates
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "events",
    "DUPLICATE_EVENT_CHECK",
    events_total,
    events_duplicate_failed
)


# ---------------------------------------------------------------------
# 13. EVENT NULL VALUE CHECK
# ---------------------------------------------------------------------

events_null_failed = spark.sql("""
    SELECT COUNT(*) AS failed_records
    FROM dbo.silver_events
    WHERE event_id IS NULL
       OR timestamp IS NULL
       OR asset_id IS NULL
       OR event_type IS NULL
       OR severity IS NULL
       OR message IS NULL
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "events",
    "NULL_VALUE_CHECK",
    events_total,
    events_null_failed
)





# ---------------------------------------------------------------------
# 15. LATE-ARRIVING EVENT CHECK
# ---------------------------------------------------------------------
# Business rule:
# Event is considered late when ingestion happens more than
# 24 hours after the actual event timestamp.
#            EVENT TIME  
            #2026-08-15 10:00
                # Data arrives
              #2026-08-18 22:00
#Event happens
#Event timestamp
# within 24 hours ──────►  Not late
# after 24 hours ───────►  Late-arriving

late_event_failed = spark.sql("""
    SELECT COUNT(*) AS failed_records
    FROM dbo.silver_events
    WHERE timestamp IS NOT NULL
      AND ingestion_timestamp IS NOT NULL
      AND ingestion_timestamp >
          timestamp + INTERVAL 24 HOURS
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "events",
    "LATE_ARRIVING_DATA_CHECK",
    events_total,
    late_event_failed
)


# =====================================================================
# ASSET METADATA VALIDATION
# =====================================================================

# ---------------------------------------------------------------------
# 16. TOTAL ASSET RECORDS
# ---------------------------------------------------------------------

asset_total = spark.sql("""
    SELECT COUNT(*) AS total_records
    FROM dbo.silver_asset_metadata
""").collect()[0]["total_records"]


# ---------------------------------------------------------------------
# 17. INCOMPLETE ASSET RECORD CHECK
# ---------------------------------------------------------------------

asset_incomplete_failed = spark.sql("""
    SELECT COUNT(*) AS failed_records
    FROM dbo.silver_asset_metadata
    WHERE asset_id IS NULL
       OR site_id IS NULL
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "asset_metadata",
    "INCOMPLETE_RECORD_CHECK",
    asset_total,
    asset_incomplete_failed
)


# ---------------------------------------------------------------------
# 18. DUPLICATE ASSET CHECK
# ---------------------------------------------------------------------
# asset_id should uniquely identify an asset.

asset_duplicate_failed = spark.sql("""
    SELECT
        COALESCE(SUM(duplicate_count - 1), 0) AS failed_records
    FROM
    (
        SELECT
            asset_id,
            COUNT(*) AS duplicate_count
        FROM dbo.silver_asset_metadata
        WHERE asset_id IS NOT NULL
        GROUP BY asset_id
        HAVING COUNT(*) > 1
    ) duplicates
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "asset_metadata",
    "DUPLICATE_ASSET_CHECK",
    asset_total,
    asset_duplicate_failed
)


# ---------------------------------------------------------------------
# 19. ASSET NULL VALUE CHECK
# ---------------------------------------------------------------------

asset_null_failed = spark.sql("""
    SELECT COUNT(*) AS failed_records
    FROM dbo.silver_asset_metadata
    WHERE asset_id IS NULL
       OR asset_name IS NULL
       OR asset_type IS NULL
       OR manufacturer IS NULL
       OR installation_date IS NULL
       OR site_id IS NULL
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "asset_metadata",
    "NULL_VALUE_CHECK",
    asset_total,
    asset_null_failed
)


# ---------------------------------------------------------------------
# 20. ASSET DATA-TYPE / STRUCTURE CHECK
# ---------------------------------------------------------------------
# installation_date must be successfully converted to DATE.
# Mandatory identifiers must also be available.

asset_schema_failed = spark.sql("""
    SELECT COUNT(*) AS failed_records
    FROM dbo.silver_asset_metadata
    WHERE asset_id IS NULL
       OR site_id IS NULL
       OR installation_date IS NULL
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "asset_metadata",
    "DATA_TYPE_CONVERSION_CHECK",
    asset_total,
    asset_schema_failed
)


# ---------------------------------------------------------------------
# 21. INVALID ASSET VALUE CHECK
# ---------------------------------------------------------------------
# Asset type must be populated.
#
# If Nectar provides an official list of allowed asset types,
# replace this check with an explicit IN (...) validation.

asset_invalid_failed = spark.sql("""
    SELECT COUNT(*) AS failed_records
    FROM dbo.silver_asset_metadata
    WHERE asset_type IS NULL
       OR TRIM(asset_type) = ''
""").collect()[0]["failed_records"]


write_dq_result(
    run_id,
    "asset_metadata",
    "INVALID_ASSET_VALUE_CHECK",
    asset_total,
    asset_invalid_failed
)


# =====================================================================
# 22. CREATE RUN-LEVEL DQ SUMMARY TABLE
# =====================================================================
# One row represents one complete DQ execution.
#
# Overall rule:
# If ANY check fails → overall_status = FAIL
# If ALL checks pass → overall_status = PASS

spark.sql("""
CREATE TABLE IF NOT EXISTS dbo.dq_quality_summary
(
    run_id STRING,
    total_checks BIGINT,
    passed_checks BIGINT,
    failed_checks BIGINT,
    overall_status STRING
)
""")


# =====================================================================
# 23. INSERT CURRENT RUN SUMMARY
# =====================================================================

spark.sql(f"""
INSERT INTO dbo.dq_quality_summary
(
    run_id,
    total_checks,
    passed_checks,
    failed_checks,
    overall_status
)

SELECT
    run_id,

    COUNT(*) AS total_checks,

    SUM(
        CASE
            WHEN status = 'PASS' THEN 1
            ELSE 0
        END
    ) AS passed_checks,

    SUM(
        CASE
            WHEN status = 'FAIL' THEN 1
            ELSE 0
        END
    ) AS failed_checks,

    CASE
        WHEN SUM(
            CASE
                WHEN status = 'FAIL' THEN 1
                ELSE 0
            END
        ) = 0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS overall_status

FROM dbo.dq_quality_report

WHERE run_id = '{run_id}'

GROUP BY run_id
""")


# =====================================================================
# 24. DISPLAY DETAILED DQ REPORT
# =====================================================================

display(
    spark.sql(f"""
        SELECT
            run_id,
            dataset_name,
            check_name,
            total_records,
            failed_records,
            passed_records,
            ROUND(failure_percentage, 2) AS failure_percentage,
            status,
            validation_timestamp

        FROM dbo.dq_quality_report

        WHERE run_id = '{run_id}'

        ORDER BY
            dataset_name,
            check_name
    """)
)




StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e74a3777-28a7-4d9a-a987-d91782f52557)

**# Create a one-row DQ summary table**

In [33]:
# ================================================================
# DQ QUALITY SUMMARY
# ================================================================
# Purpose:
#   Creates one summary record for each DQ execution.
#
#   PASS -> All DQ checks passed
#   FAIL -> At least one DQ check failed
# ================================================================

display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.dq_quality_summary AS

        SELECT
            run_id,

            -- Total number of DQ checks executed
            COUNT(*) AS total_checks,

            -- Number of successful checks
            SUM(
                CASE
                    WHEN status = 'PASS' THEN 1
                    ELSE 0
                END
            ) AS passed_checks,

            -- Number of failed checks
            SUM(
                CASE
                    WHEN status = 'FAIL' THEN 1
                    ELSE 0
                END
            ) AS failed_checks,

            -- Overall DQ status
            CASE
                WHEN SUM(
                    CASE
                        WHEN status = 'FAIL' THEN 1
                        ELSE 0
                    END
                ) = 0
                THEN 'PASS'
                ELSE 'FAIL'
            END AS overall_status,

            -- Time of the DQ execution
            MAX(validation_timestamp) AS validation_timestamp

        FROM dbo.dq_quality_report

        GROUP BY run_id
    """)
)




StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b962fe21-d84c-4d68-b87c-0b24257b13d1)

In [34]:
# ================================================================
# DISPLAY DQ SUMMARY
# ================================================================
# Latest DQ execution appears first.

display(
    spark.sql("""
        SELECT
            run_id,
            total_checks,
            passed_checks,
            failed_checks,
            overall_status,
            validation_timestamp

        FROM dbo.dq_quality_summary

        ORDER BY validation_timestamp DESC
    """)
)

StatementMeta(, 11e3596e-bebe-4fe3-9409-d47525bf84cd, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4bce3fa4-5f8b-4dcb-af33-632ad5bda1e2)